# E2E Demo Test Runner

Validates every module demo in the current training environment.
Run **top-to-bottom** before each session.

| Module | Coverage |
|--------|----------|
| Setup  | catalog + schemas + Volume |
| M01    | SparkSession, Delta table, Widgets |
| M02    | Lazy eval, AQE, Catalyst plan |
| M03    | CSV/JSON schema, transformations, JOIN |
| M04    | CRUD, schema evolution, Time Travel, RESTORE |
| M05    | Widget types, dynamic params, system tables |
| M06    | Catalog audit, table tags |
| M07    | Gold layer build, BI query |
| Teardown | DROP all test tables |


## Setup

In [ ]:
%run ../setup/00_setup
print(f'Catalog:      {CATALOG}')
print(f'Dataset path: {DATASET_PATH}')

In [ ]:
schemas = [r[0] for r in spark.sql(f'SHOW SCHEMAS IN {CATALOG}').collect()]
for s in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    assert s in schemas, f'Missing schema: {s}'
    print(f'  [OK] {CATALOG}.{s}')
print('\n✓ Environment ready')

## M01 — Platform & Workspace

In [ ]:
from pyspark.sql import functions as F
print(f'Spark version: {spark.version}')
print(f'Current user:  {spark.sql("SELECT current_user()").first()[0]}')
print('✓ M01-D1 SparkSession OK')

In [ ]:
df_demo = spark.createDataFrame(
    [('Alice','PL',1000),('Bob','DE',2500),('Carol','FR',1800)],
    ['name','country','revenue'])
df_demo.createOrReplaceTempView('demo_people')
assert df_demo.count() == 3
print('✓ M01-D2 DataFrame + temp view')

In [ ]:
spark.sql(f'DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.test_m01_demo')
(df_demo.write.format('delta').mode('overwrite')
    .saveAsTable(f'{CATALOG}.{BRONZE_SCHEMA}.test_m01_demo'))
cnt = spark.table(f'{CATALOG}.{BRONZE_SCHEMA}.test_m01_demo').count()
assert cnt == 3
print(f'✓ M01-D3 Delta table — {cnt} rows')

In [ ]:
dbutils.widgets.text('test_w','hello','Test')
assert dbutils.widgets.get('test_w') == 'hello'
dbutils.widgets.removeAll()
print('✓ M01-D4 Widget OK')

## M02 — Spark Architecture

In [ ]:
df_ny = spark.table('samples.nyctaxi.trips')
df_plan = (df_ny
    .filter(F.col('trip_distance') > 1)
    .withColumn('km', F.round(F.col('trip_distance')*1.609, 2))
    .withColumn('cost_per_km', F.round(F.col('fare_amount')/F.col('km'),2)))
plan_str = df_plan._jdf.queryExecution().toString()
assert len(plan_str) > 0
print('✓ M02-D1 Lazy eval — plan built without action')

In [ ]:
cnt = df_plan.count()
assert cnt > 0
print(f'✓ M02-D2 COUNT action executed -> {cnt:,} rows')

In [ ]:
aqe = spark.conf.get('spark.sql.adaptive.execution.enabled','false')
print(f'AQE enabled: {aqe}')
print('✓ M02-D3 AQE config OK')

In [ ]:
df_api = (spark.table('samples.nyctaxi.trips')
    .groupBy('payment_type')
    .agg(F.round(F.avg('fare_amount'),2).alias('avg_fare'))
    .orderBy('payment_type'))
df_sql = spark.sql("""
    SELECT payment_type, ROUND(AVG(fare_amount),2) AS avg_fare
    FROM samples.nyctaxi.trips
    GROUP BY payment_type ORDER BY payment_type""")
assert df_api.count() == df_sql.count()
print(f'✓ M02-D4 PySpark == SQL ({df_api.count()} payment types)')

## M03 — Medallion Ingestion

In [ ]:
from pyspark.sql.types import (StructType,StructField,StringType,
    IntegerType,DoubleType,DateType,TimestampType)

customer_schema = StructType([
    StructField('customer_id',       StringType(),  False),
    StructField('first_name',        StringType(),  True),
    StructField('last_name',         StringType(),  True),
    StructField('email',             StringType(),  True),
    StructField('country',           StringType(),  True),
    StructField('registration_date', DateType(),    True),
    StructField('loyalty_points',    IntegerType(), True),
    StructField('annual_spend',      DoubleType(),  True),
])
df_customers = (spark.read
    .schema(customer_schema)
    .option('header','true')
    .option('dateFormat','yyyy-MM-dd')
    .csv(f'{DATASET_PATH}/customers/customers.csv'))
assert dict(df_customers.dtypes)['registration_date'] == 'date'
print(f'✓ M03-D1 customers.csv -> {df_customers.count():,} rows')

In [ ]:
orders_schema = StructType([
    StructField('order_id',         StringType(),    False),
    StructField('customer_id',      StringType(),    True),
    StructField('product_id',       StringType(),    True),
    StructField('store_id',         StringType(),    True),
    StructField('order_datetime',   TimestampType(), True),
    StructField('quantity',         IntegerType(),   True),
    StructField('unit_price',       DoubleType(),    True),
    StructField('discount_percent', DoubleType(),    True),
    StructField('total_amount',     DoubleType(),    True),
    StructField('payment_method',   StringType(),    True),
    StructField('status',           StringType(),    True),
])
df_orders = (spark.read
    .schema(orders_schema)
    .option('timestampFormat','yyyy-MM-dd HH:mm:ss')
    .json(f'{DATASET_PATH}/orders/orders_batch.json'))
assert dict(df_orders.dtypes)['order_datetime'] == 'timestamp'
print(f'✓ M03-D2 orders_batch.json -> {df_orders.count():,} rows')

In [ ]:
df_top = (df_customers.filter(F.col('loyalty_points')>=500)
    .groupBy('country')
    .agg(F.count('customer_id').alias('n'),
         F.round(F.avg('annual_spend'),2).alias('avg'))
    .orderBy(F.col('avg').desc()))
assert df_top.count() > 0
df_joined = df_orders.join(df_customers, on='customer_id', how='left')
assert df_joined.count() == df_orders.count()
print(f'✓ M03-D3 groupBy + JOIN -> {df_joined.count():,} rows')

In [ ]:
df_customers.createOrReplaceTempView('tmp_customers')
df_orders.createOrReplaceTempView('tmp_orders')
r = spark.sql("""
    SELECT c.country, COUNT(o.order_id) AS orders
    FROM tmp_orders o LEFT JOIN tmp_customers c ON o.customer_id=c.customer_id
    GROUP BY c.country ORDER BY orders DESC LIMIT 5""")
assert r.count() > 0
r.show()
print('✓ M03-D4 SQL temp view query OK')

## M04 — Delta Lake Fundamentals

In [ ]:
T = f'{CATALOG}.{BRONZE_SCHEMA}.test_m04_customers'
spark.sql(f'DROP TABLE IF EXISTS {T}')
(df_customers.limit(100).write.format('delta').mode('overwrite')
    .option('overwriteSchema','true').saveAsTable(T))
assert spark.table(T).count() == 100
print(f'✓ M04-D1 Delta table created (v0)')

In [ ]:
spark.sql(f"""
    INSERT INTO {T} (customer_id,first_name,last_name,email,country,loyalty_points,annual_spend)
    VALUES ('CUST9999','Test','User','t@x.com','PL',0,0.0)""")
assert spark.table(T).count() == 101
print('✓ M04-D2 INSERT -> v1')

In [ ]:
spark.sql(f"UPDATE {T} SET loyalty_points=9999 WHERE customer_id='CUST9999'")
lp = spark.table(T).filter("customer_id='CUST9999'").select('loyalty_points').first()[0]
assert lp == 9999
print(f'✓ M04-D3 UPDATE -> loyalty_points={lp}')

In [ ]:
spark.sql(f"DELETE FROM {T} WHERE customer_id='CUST9999'")
assert spark.table(T).count() == 100
print('✓ M04-D4 DELETE -> back to 100 rows')

In [ ]:
history = spark.sql(f'DESCRIBE HISTORY {T}').select('version','operation').collect()
ops = [h.operation for h in history]
print('Versions:', [(h.version, h.operation) for h in history])
assert len(history) >= 3
print(f'✓ M04-D5 DESCRIBE HISTORY -> {len(history)} versions')

In [ ]:
df_v1 = spark.sql(f'SELECT * FROM {T} VERSION AS OF 1')
assert df_v1.count() == 101
print(f'✓ M04-D6 Time Travel v1 -> {df_v1.count()} rows')

In [ ]:
spark.sql(f'RESTORE TABLE {T} TO VERSION AS OF 1')
assert spark.table(T).count() == 101
spark.sql(f'RESTORE TABLE {T} TO VERSION AS OF 3')
assert spark.table(T).count() == 100
print('✓ M04-D7 RESTORE v1 then v3 OK')

In [ ]:
from pyspark.sql.types import StructType,StructField,StringType
df_extra = spark.createDataFrame([('CUST0001','gold')],['customer_id','customer_tier'])
(df_extra.write.format('delta').mode('append')
    .option('mergeSchema','true').saveAsTable(T))
assert 'customer_tier' in spark.table(T).columns
print('✓ M04-D8 Schema evolution mergeSchema OK')

## M05 — Orchestration

In [ ]:
dbutils.widgets.text('env','dev','Environment')
dbutils.widgets.dropdown('region','EU',['EU','US','APAC'],'Region')
dbutils.widgets.combobox('table','orders',['orders','customers'],'Table')
dbutils.widgets.multiselect('cols','id',['id','name','date','amount'],'Cols')
assert dbutils.widgets.get('env') == 'dev'
assert dbutils.widgets.get('region') == 'EU'
dbutils.widgets.removeAll()
print('✓ M05-D1 All widget types created and read OK')

In [ ]:
df_runs = spark.sql("""
    SELECT * FROM system.lakeflow.job_run_timeline
    ORDER BY 1 DESC LIMIT 5""")
print(f'✓ M05-D2 system.lakeflow accessible -> {df_runs.count()} recent rows')

## M06 — Unity Catalog

In [ ]:
catalogs = [r[0] for r in spark.sql('SHOW CATALOGS').collect()]
schemas  = [r[0] for r in spark.sql(f'SHOW SCHEMAS IN {CATALOG}').collect()]
assert CATALOG in catalogs
assert BRONZE_SCHEMA in schemas
print(f'✓ M06-D1 Catalog audit OK — {len(catalogs)} catalogs, schemas={schemas}')

In [ ]:
privs = spark.sql(f"""
    SELECT grantee,privilege_type,object_name,object_type
    FROM {CATALOG}.information_schema.table_privileges LIMIT 5""")
print(f'✓ M06-D2 information_schema.table_privileges -> {privs.count()} rows')
privs.show(truncate=False)

In [ ]:
demo_tbl = f'{CATALOG}.{BRONZE_SCHEMA}.test_m01_demo'
spark.sql(f"ALTER TABLE {demo_tbl} SET TAGS ('env'='test','owner'='trainer')")
tags = spark.sql(f'SHOW TBLPROPERTIES {demo_tbl}').filter("key LIKE 'tag.%'")
print(f'✓ M06-D3 Table tags -> {tags.count()} tag(s)')
tags.show(truncate=False)

## M07 — AI/BI Dashboards

In [ ]:
TABLE_GOLD = f'{CATALOG}.{GOLD_SCHEMA}.test_m07_orders_summary'
spark.sql(f'DROP TABLE IF EXISTS {TABLE_GOLD}')
df_gold = (df_orders
    .filter(F.col('status').isin(['shipped','delivered']))
    .withColumn('order_date', F.to_date(F.col('order_datetime').cast('timestamp')))
    .groupBy('order_date','payment_method')
    .agg(F.count('order_id').alias('total_orders'),
         F.round(F.sum('total_amount'),2).alias('total_revenue'),
         F.round(F.avg('total_amount'),2).alias('avg_order_value'))
    .orderBy('order_date'))
(df_gold.write.format('delta').mode('overwrite').saveAsTable(TABLE_GOLD))
print(f'✓ M07-D1 Gold summary -> {spark.table(TABLE_GOLD).count()} rows')

In [ ]:
r = spark.sql(f"""
    SELECT order_date, SUM(total_revenue) AS rev, SUM(total_orders) AS orders
    FROM {TABLE_GOLD}
    GROUP BY order_date ORDER BY rev DESC LIMIT 5""")
assert r.count() == 5
r.show()
print('✓ M07-D2 BI-ready query OK')

## Teardown

In [ ]:
for tbl in [
    f'{CATALOG}.{BRONZE_SCHEMA}.test_m01_demo',
    f'{CATALOG}.{BRONZE_SCHEMA}.test_m04_customers',
    f'{CATALOG}.{GOLD_SCHEMA}.test_m07_orders_summary',
]:
    spark.sql(f'DROP TABLE IF EXISTS {tbl}')
    print(f'  dropped: {tbl}')
print('✓ Teardown complete')

## Summary
If all cells printed ✓, the training environment is fully ready.

Modules covered: Setup, M01-M07
